# Memorization Type 3: Subtrajectory Memorization

This notebook is part of the broader memorization analysis framework for mobility sequence predictors.
The focus here is on understanding **which parts of a trajectory are being memorized** by the model —
in particular, short sub-sequences of user behavior.

### Memorization Type 3: What we test

We aim to evaluate whether **specific segments** of a trajectory (e.g., nighttime routine, work commute, leisure period)
have been memorized by the model.

In other words, we take a full training trajectory and systematically **replace short segments** of it
with realistic alternatives, and observe how much the model’s perplexity changes.

If replacing a small portion leads to a large drop in likelihood, this suggests **localized memorization**.

### This notebook performs the following:
1. Load and preprocess full user trajectories.
2. For each training trajectory:
   - Define sliding windows over the trajectory (e.g., every 4 hours).
   - Generate realistic alternative segments via:
       • Repeated patterns from the same user (other days),
       • Common subtrajectories from other users,
       • Synthetic neutral behavior (e.g., stationarity).
   - Replace the original segment with each alternative and reconstruct full sequences.
3. Evaluate the model’s log-likelihood / perplexity on:
   - Original trajectory,
   - Each substituted version (reference set).
4. Measure and visualize memorization metrics at the **subtrajectory level**.

### Objective
This analysis enables us to ask:
- Which *parts* of a user’s trajectory are most memorized?
- Are some subtrajectories (e.g., daily commutes) more likely to be memorized than others?
- Can we quantify memorization *locally* — at the scale of just a few hours?


In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
from haversine import haversine
import random
sys.path.append('./Helpers/')
from utils import save_dict, load_dict
from sklearn.cluster import AgglomerativeClustering
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from collections import defaultdict
from sklearn.cluster import KMeans
from fastdtw import fastdtw
from scipy.spatial.distance import squareform
from sklearn.cluster import AgglomerativeClustering
import numpy as np
from tqdm import tqdm  

In [22]:
DATASET_NAME_TO_FOLDER_NAME = {
    'boston': 'Boston',
    'geolife': 'Geolife',
    'shenzhenurban': 'ShenzhenUrban',
    'shanghaikaggle': 'ShanghaiKaggle',
    'yjmob100k': 'YJMob100Kv3'
}
dataset_name = "yjmob100k"
version=0
save_path_data_datasets = (
    Path('PreprocessedData/2SplittedData') /
    DATASET_NAME_TO_FOLDER_NAME[dataset_name] /
    'NormalizationType3' / 
    'Datasets/'
)


## Dataset abstraction

In [23]:
df = pd.read_pickle(Path('PreprocessedData/1FilteredData/') / DATASET_NAME_TO_FOLDER_NAME[dataset_name]/str(version)/'dataset.pkl', compression=None)
location_df = pd.read_csv(Path('PreprocessedData/1FilteredData/') / DATASET_NAME_TO_FOLDER_NAME[dataset_name]/str(version)/'dictionary.csv')
location_dict = location_df.set_index('Location')[['Latitude', 'Longitude']].apply(tuple, axis=1).to_dict()

In [24]:
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['date'] = df['Timestamp'].dt.date
df['Latitude'] = df['Location'].map(location_dict).str[0]
df['Longitude'] = df['Location'].map(location_dict).str[1]
traj_df = df[['DeviceID', 'date']].drop_duplicates().reset_index().rename(columns={'index': 'traj_id'})
df = df.merge(right=traj_df, on=['DeviceID', 'date'])

In [25]:
df.DeviceID.nunique()

99610

In [26]:
def generate_type3_datasets(df, datasets_path, k=3, window_hours=4, n_reference=100):
    datasets_path = Path(datasets_path)
    datasets_path.mkdir(parents=True, exist_ok=True)

    expected_len = 48 * k
    steps_per_hour = 2
    window_size = window_hours * steps_per_hour
    num_windows = expected_len // window_size
    n_per_window = n_reference // num_windows

    training_rows = []
    metadata_rows = []

    df = df.sort_values('Timestamp')

    for device_id, user_df in tqdm(df.groupby('DeviceID'), total=df['DeviceID'].nunique(), desc="Processing users"):
        user_df = user_df.sort_values('Timestamp')
        unique_trajs = sorted(user_df['traj_id'].unique())

        # Try to extract first complete k-day sequence
        for i in range(0, len(unique_trajs) - k + 1):
            selected_trajs = unique_trajs[i:i + k]
            chunk_df = user_df[user_df['traj_id'].isin(selected_trajs)].sort_values('Timestamp')
            if len(chunk_df) == expected_len:
                break
        else:
            continue  # No complete k-day chunk

        training_df = chunk_df.copy()
        training_df['tid'] = f"{device_id}_{selected_trajs[0]}_{selected_trajs[-1]}"
        training_tid = training_df['tid'].iloc[0]
        training_rows.append(training_df)

        coords = training_df[['Latitude', 'Longitude']].to_numpy()
        reference_dfs = []

        for win in range(num_windows):
            start_idx = win * window_size
            window_slice = slice(start_idx, start_idx + window_size)

            for rep in range(n_per_window):
                pert_type = random.choice(['shuffle', 'stationary', 'substitute'])
                new_coords = coords.copy()

                if pert_type == 'shuffle':
                    sub = new_coords[window_slice]
                    np.random.shuffle(sub)
                    new_coords[window_slice] = sub

                elif pert_type == 'stationary':
                    # pick a random stationary location from the window
                    sub = new_coords[window_slice]
                    mask = ~np.isnan(sub).any(axis=1)
                    if not mask.any():
                        continue
                    candidate = sub[random.choice(np.where(mask)[0])]
                    new_coords[window_slice] = np.tile(candidate, (window_size, 1))

                # elif pert_type == 'substitute':
                #     # Pick another traj (not part of selected_trajs)
                #     available = [t for t in unique_trajs if t not in selected_trajs]
                #     if not available:
                #         continue
                #     alt_tid = random.choice(available)
                #     alt_df = user_df[user_df['traj_id'] == alt_tid].sort_values('Timestamp')
                #     alt_coords = alt_df[['Latitude', 'Longitude']].to_numpy()
                #     if len(alt_coords) < window_size:
                #         continue
                #     start_alt = random.randint(0, len(alt_coords) - window_size)
                #     new_coords[window_slice] = alt_coords[start_alt:start_alt + window_size]

                elif pert_type == 'substitute':
                    available = [t for t in unique_trajs if t not in selected_trajs]
                    if not available:
                        continue
                    alt_tid = random.choice(available)
                    alt_df = user_df[user_df['traj_id'] == alt_tid].sort_values('Timestamp')
                    alt_coords = alt_df[['Latitude', 'Longitude']].to_numpy()
                    alt_timestamps = alt_df['Timestamp'].to_numpy()
                    
                    if len(alt_coords) < window_size:
                        continue

                    # Get hour of day for start of the current window
                    original_start_hour = training_df['Timestamp'].iloc[start_idx].hour

                    # Find matching windows in alt_df that start at the same hour
                    candidates = []
                    for i in range(len(alt_coords) - window_size):
                        candidate_hour = pd.Timestamp(alt_timestamps[i]).hour
                        if candidate_hour == original_start_hour:
                            candidates.append(i)
                    
                    if not candidates:
                        continue
                    
                    start_alt = random.choice(candidates)
                    new_coords[window_slice] = alt_coords[start_alt:start_alt + window_size]
                ref_df = training_df.copy()
                ref_df[['Latitude', 'Longitude']] = new_coords
                ref_tid = f"{training_tid}_ref_win{win}_v{rep}"
                ref_df['tid'] = ref_tid
                reference_dfs.append(ref_df)

                metadata_rows.append({
                    'device_id': device_id,
                    'training_tid': training_tid,
                    'reference_tid': ref_tid,
                    'perturbation': pert_type,
                    'window_index': win,
                    'window_start_hour': (start_idx // 2),
                    'window_size_hours': window_hours
                })

        # Save per-user reference dataset
        if reference_dfs:
            user_ref_df = pd.concat(reference_dfs)
            user_ref_df = user_ref_df[['tid', 'Timestamp', 'Latitude', 'Longitude']]
            user_ref_df = user_ref_df.rename(columns={'Timestamp': 'timestamp', 'Latitude': 'lat', 'Longitude': 'lon'})
            user_ref_df[['lat', 'lon']] = user_ref_df.groupby('tid')[['lat', 'lon']].transform(lambda g: g.ffill())
            user_ref_df.to_csv(datasets_path / f"{device_id}.csv", index=False)

    # Save training set
    if training_rows:
        training_df = pd.concat(training_rows)
        training_df = training_df[['tid', 'Timestamp', 'Latitude', 'Longitude']]
        training_df = training_df.rename(columns={'Timestamp': 'timestamp', 'Latitude': 'lat', 'Longitude': 'lon'})
        training_df[['lat', 'lon']] = training_df.groupby('tid')[['lat', 'lon']].transform(lambda g: g.ffill())
        training_df.to_csv(datasets_path / "training_set.csv", index=False)

    # Save metadata
    if metadata_rows:
        metadata_df = pd.DataFrame(metadata_rows)
        metadata_df.to_csv(datasets_path / "representant_mapping.txt", index=False)

    print(f"\n✅ Done. Training set and {len(metadata_rows)} reference trajectories saved.")


In [27]:
sampled_ids = pd.Series(df['DeviceID'].unique()).sample(min(df['DeviceID'].nunique(), 2000), random_state=42)
df_sample = df[df['DeviceID'].isin(sampled_ids)]


In [28]:
generate_type3_datasets(df_sample, save_path_data_datasets, k=3, window_hours=4, n_reference=300)

Processing users: 100%|████████████████████████████████████████████| 2000/2000 [13:01<00:00,  2.56it/s]



✅ Done. Training set and 544058 reference trajectories saved.
